# 3D change detection between two phone videos

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ellecarey/humanoid/blob/main/notebooks/run_on_colab.ipynb)

Use a GPU runtime (A100, L4 or larger). Run the cells in order: install, check the GPU, pick the videos, optionally set a Gemini key, run, inspect the debug output, then view the report.

In [ ]:
import os

if not os.path.isdir("/content/humanoid"):
    !git clone --depth 1 https://github.com/ellecarey/humanoid.git /content/humanoid
else:
    !git -C /content/humanoid pull --ff-only
%cd /content/humanoid

In [ ]:
%pip install -q -e ".[gpu,vlm]"

In [ ]:
from pathlib import Path

import torch
import yaml

if not torch.cuda.is_available():
    raise SystemExit("No GPU: switch the runtime type to a GPU and rerun.")
props = torch.cuda.get_device_properties(0)
memory_gb = props.total_memory / 1e9
# MapAnything's memory-efficient mode scales to hundreds of views; these budgets leave headroom.
per_video = 100 if memory_gb >= 40 else 60 if memory_gb >= 22 else 35
print(f"{props.name}: {memory_gb:.0f} GB -> frames.max_per_video = {per_video}")

config = yaml.safe_load(Path("configs/default.yaml").read_text())
config["frames"]["max_per_video"] = per_video
Path("configs/colab.yaml").write_text(yaml.safe_dump(config, sort_keys=False))

In [ ]:
# Optional: keep stage caches on Drive so a disconnect does not lose the reconstruction.
import shutil

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    cache = Path("/content/drive/MyDrive/humanoid_outputs")
    cache.mkdir(parents=True, exist_ok=True)
    outputs = Path("outputs")
    if outputs.exists() and not outputs.is_symlink():
        shutil.rmtree(outputs)
    if not outputs.exists():
        outputs.symlink_to(cache)
    print(f"outputs/ -> {cache}")

In [ ]:
dest = Path("data/raw_videos")
dest.mkdir(parents=True, exist_ok=True)
v1, v2 = dest / "scene_v1.MOV", dest / "scene_v2.MOV"

clips = sorted(
    path
    for path in Path("examples").glob("*")
    if path.suffix.lower() in {".mp4", ".mov"}
)
if v1.is_file() and v2.is_file():
    print(f"Using {v1} and {v2}")
elif len(clips) >= 2:
    shutil.copy(clips[0], v1)
    shutil.copy(clips[1], v2)
    print(f"Using example clips {clips[0].name} and {clips[1].name}")
else:
    from google.colab import files

    uploaded = sorted(files.upload())
    if len(uploaded) != 2:
        raise SystemExit("Upload exactly two videos, v1 then v2.")
    for name, target in zip(uploaded, (v1, v2)):
        Path(name).replace(target)
    print(f"Uploaded {v1.name} and {v2.name}")

In [ ]:
from getpass import getpass

key = ""
try:
    from google.colab import userdata

    key = userdata.get("GEMINI_API_KEY") or ""
except Exception:
    pass
if not key:
    key = getpass("GEMINI_API_KEY (leave blank to use offline labels): ").strip()
if key:
    os.environ["GEMINI_API_KEY"] = key

In [ ]:
!humanoid run --config configs/colab.yaml --force

## Debug output

Check that the frame sheets cover the whole room in both videos and that `overlap_after` in the alignment stats is above about 0.7. `scores.html` holds the vote and appearance histograms used to set the thresholds.

In [ ]:
import json

from IPython.display import HTML, Image, display

debug = Path("outputs/demo/debug")
for sheet in sorted(debug.glob("frames_*.jpg")):
    print(sheet.name)
    display(Image(str(sheet), width=1200))
for name in ("alignment.json", "detect.json", "ground_truth.json"):
    path = debug / name
    if path.is_file():
        print(f"--- {name}")
        print(json.dumps(json.loads(path.read_text()), indent=1)[:4000])
for crop in sorted((debug / "gemini").glob("region_*.jpg")):
    print(crop.name)
    display(Image(str(crop), width=360))
if (debug / "scores.html").is_file():
    display(HTML((debug / "scores.html").read_text()))

In [ ]:
# Tune: change thresholds, then rerun only the light stages from the cached reconstruction.
config = yaml.safe_load(Path("configs/colab.yaml").read_text())
config["visibility"].update(
    {"min_free_views": 3, "free_ratio": 0.6, "appearance_cos": 0.55}
)
Path("configs/colab.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
!humanoid run --config configs/colab.yaml --from-stage detect

In [ ]:
display(HTML(Path("outputs/demo/report/report.html").read_text()))